In [2]:
import random
import pandas as pd
import base64
import json

# -----------------------------
# 1. Participants
# -----------------------------
names = ["Alba","Ari","Paolo","Dani","Grazia","Moreno","Edo","Flavio",
         "Sonia","Pippo","Marci","Andrea","Silvia","Giada","Johnny",
         "Nonna Ilda","Nonna Elena"]

codes_unshuffled = ["PK94","GT33","MN23","PL87","RC21","VR46",
                    "PC45","RM28","FE12","CT36","FI09","TN57",
                    "BS78","NA34","TO67","MD85","ME94"]

# -----------------------------
# 2. Shuffle codes but assign row-wise
# -----------------------------
codes_shuffled = codes_unshuffled[:]
random.shuffle(codes_shuffled)

df = pd.DataFrame(names, columns=["name"])
df["code"] = codes_shuffled  # assign codes row by row (preserves mapping)

# -----------------------------
# 3. Last year assignment (fixed cycle)
# -----------------------------
names_last_year_str = "Sonia>Nonna Ilda>Silvia>Pippo>Paolo>Edo>Flavio>Moreno>Nonna Elena>Grazia>Ari>Johnny>Andrea>Dani>Marci>Giada>Alba"
names_last_year = [n.strip() for n in names_last_year_str.split(">")]

def create_cycle(names, last_year):
    index_map = {name: i for i, name in enumerate(names)}
    last_year_indices = [index_map[name] for name in last_year]
    cycle = [0] * len(names)
    for i in range(len(names)):
        current_index = last_year_indices[i]
        next_index = last_year_indices[(i + 1) % len(names)]
        cycle[current_index] = next_index
    return [names[i] for i in cycle]

df["last_year"] = create_cycle(names, names_last_year)

# -----------------------------
# 4. This year assignment (derangement)
# -----------------------------
def single_cycle_list(lst):
    lst = lst[:]
    random.shuffle(lst)
    return lst

this_year = single_cycle_list(names)
df["this_year"] = this_year

# ensure nobody gets same as last year and nobody gets themselves
while not all(df["this_year"][i] != df["last_year"][i] and df["this_year"][i] != df["name"][i] for i in range(len(names))):
    this_year = single_cycle_list(names)
    df["this_year"] = this_year

# -----------------------------
# 5. Generate JSON
# -----------------------------
assignments = {}
for _, row in df.iterrows():
    assignments[row["code"]] = {
        "name": row["name"],
        "giftee": base64.b64encode(row["this_year"].encode()).decode()
    }

# -----------------------------
# 6. Save JSON for embedding
# -----------------------------
with open("assignments.json", "w") as f:
    json.dump(assignments, f, indent=2)

# Optional: verify correct mapping
for code, data in assignments.items():
    print( "{} ha il codice {} ".format(data["name"], code) )


Alba ha il codice PC45 
Ari ha il codice CT36 
Paolo ha il codice RM28 
Dani ha il codice TO67 
Grazia ha il codice PL87 
Moreno ha il codice PK94 
Edo ha il codice ME94 
Flavio ha il codice BS78 
Sonia ha il codice NA34 
Pippo ha il codice RC21 
Marci ha il codice MN23 
Andrea ha il codice MD85 
Silvia ha il codice GT33 
Giada ha il codice FE12 
Johnny ha il codice VR46 
Nonna Ilda ha il codice TN57 
Nonna Elena ha il codice FI09 
